# Spare-It POC: Synthetic Image Generator

In [3]:
import os
import json
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from pycocotools.coco import COCO
from matplotlib import image
from PIL import Image
from scipy import ndimage
import math
from skimage import measure
from imantics import Polygons, Mask

In [4]:
# Methods for image cropping and masking
def crop(arr):
    slice_x, slice_y = ndimage.find_objects(arr>0)[0]
    return arr[slice_x, slice_y]
def cropc(arr):
    slice_x, slice_y, slice_z = ndimage.find_objects(arr>0)[0]
    return arr[slice_x, slice_y, slice_z]
def crop_coord(arr):
    slice_x, slice_y = ndimage.find_objects(arr>0)[0]
    return [slice_x, slice_y]
def apply_mask(image, mask):
    # Convert to numpy arrays
    mask = np.array(mask)
    # Convert grayscale image to RGB
    mask = np.stack((mask,)*3, axis=-1)
    # Multiply arrays
    resultant = image*mask
    return resultant

In [5]:
# Method for filenaming convention
# Finds current highest number filename in output directory and names the next file one more than it
def max_file(path):
    files = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]
    max = 0
    for f in files:
        s = f.split('.')
        if(int(s[0]) > max):
            max = int(s[0])
    return max + 1

In [6]:
# Method for sampling a normally distributed random variable for a given file dimension
def random_offset(dist):
    center = dist/2
    var = dist/10
    rand = math.ceil(np.random.normal(center, var, 1)[0])
    if(rand < dist/4 or rand > 3*dist/4):
        return math.ceil(center)
    else:
        return rand

In [7]:
# Copy Paste Method
# Takes a list containing lists with a name of a json file and the category ID to be used with that json
# It also takes a base json file
# The part of each file that has its relevant category ID is pasted as chunks onto the base image with random positioning
# Returns the augmented image, json file, and mask
def copypaste(files, file2):
    dataDir='./cocojson' # Source of ground truth json files
    img_dir = './images' # Source of image files
    dataType='val'
    coco2=COCO('{}/'.format(dataDir,dataType) + file2) # COCO filetype for base image
    img2 = coco2.imgs[0] # Base image as described in json file
    image2 = np.array(Image.open(os.path.join(img_dir, img2.get('file_name').split('images/')[-1]))) # Array representing base image
    cat_ids = coco2.getCatIds()
    anns_ids2 = coco2.getAnnIds(imgIds=img2['id'], catIds=cat_ids, iscrowd=None)
    anns2 = coco2.loadAnns(anns_ids2) # List of annotations for base image
    mask2 = np.zeros((img2['height'],img2['width']))
    
    for i in range(len(anns2)):
        # Iterates over annotations and adds them to the mask
        # Each pixel is the ID of the category from the annotations
        temp = coco2.annToMask(anns2[i])*anns2[i]['category_id']
        mask2 = np.where(temp != 0, temp, mask2)
        
    for i in files:
        # Iterates over each file in the list of jsons
        file1 = i[0]
        cid = i[1]
        coco1=COCO('{}/'.format(dataDir,dataType) + file1) # COCO filetype for currently pasted image
        img1 = coco1.imgs[0]
        image1 = np.array(Image.open(os.path.join(img_dir, img1.get('file_name').split('images/')[-1])))
        anns_ids1 = coco1.getAnnIds(imgIds=img1['id'], catIds=cat_ids, iscrowd=None)
        anns1 = coco1.loadAnns(anns_ids1)
        mask1 = np.zeros((img1['height'],img1['width']))
        
        for i in range(len(anns1)):
            temp = coco1.annToMask(anns1[i])*anns1[i]['category_id']
            mask1 = np.where(temp != 0, temp, mask1)

        # Crops the image and mask for the current file being pasted to only include the selected category
        paste = np.where(mask1 == cid, 1, 0)
        mask_cropped = crop(paste)*cid
        image_cropped = cropc(apply_mask(image1, paste))

        # Gets a random offset for the x and y coordinate based on the base image file dimensions
        x_offset=random_offset(img2['width'])
        y_offset=random_offset(img2['height'])

        # Nested for loops that go pixel by pixel to place the pasted chunk onto the base image
        # Iterates over the dimensions of where the chunk will be pasted
        # Increments track where in the pasted chunk the loop is
        y_inc = 0
        for i in range(y_offset,y_offset+image_cropped.shape[0]):
            x_inc = 0
            if(i < img2['height']): # Ensures that the chunk is not pasted outside of the base image bounds
                for j in range(x_offset,x_offset+image_cropped.shape[1]):
                    if(mask_cropped[y_inc, x_inc] != 0 and j < img2['width']):
                        image2[i, j, :] = image_cropped[y_inc, x_inc, :]
                        mask2[i,j] = mask_cropped[y_inc, x_inc] # The mask is also updated with the pasted chunk
                    x_inc +=1
            y_inc += 1
    filenum = max_file('./synthetic_images') # Generates serial number to be the name for all exported files 
    newImg = Image.fromarray(image2)
    newImg.save('./synthetic_images/' + str(filenum) + '.jpeg') # Exports augmented image
    np.save('./masks/' + str(filenum), mask2) # Saves augmented mask as numpy array
    annotations = get_annotations(mask2) # Converts mask to COCO format annotations
    # Copies and exports the original json file with the annotations section updated
    with open('./cocojson/' + file2, 'r+') as file:
        template = json.load(file) 
    template['annotations'] = annotations
    with open('./synthetic_jsons/' + str(filenum) + '.json', 'w') as f:
        json.dump(template, f, indent=4)

In [ ]:
# Converts a mask with multiple IDs to a list of binary masks with the related ID kept in a list
def mask_to_binary(mask):
    vals = np.unique(mask)
    z_index = np.where(vals == 0)
    vals = np.delete(vals, z_index)
    n = len(vals)
    binaries = []
    for i in range(n):
        binary = np.zeros(np.shape(mask))
        binary = np.where(mask == vals[i], 1, 0)
        binaries.append(binary)
    return binaries, vals
# Converts a mask with multiple IDs to COCO format annotations
def get_annotations(mask):
    bins, cids = mask_to_binary(mask)
    n = len(cids)
    annotations = []
    inc = 0
    for i in range(n):
        polygons = Mask(bins[i]).polygons().segmentation # Generates polygonal sets of coordinates for the mask
        for j in range(len(polygons)):
            ann = {}
            ann['id'] = inc
            inc += 1
            ann['image_id'] = 0
            ann['category_id'] = int(cids[i])
            polygon = polygons[j]
            seg = [polygon]
            ann['segmentation'] = seg
            annotations.append(ann)
    return annotations
            



# Loads all files into dictionary of sets
files = [f for f in os.listdir('./cocojson/') if os.path.isfile(os.path.join('./cocojson/', f))]
with open('./cocojson/' + files[-1], 'r+') as file:
        template = json.load(file)
cats = template['categories']
categories = {}
for i in cats:
    categories[i['id']] = i['name']
files_by_id = {}
files_by_id2 = {}
for i in categories:
    files_by_id[i] = set()
    files_by_id2[i] = set()

inc = 0
for i in files:
    with open('./cocojson/' + i, 'r+') as file:
        temp = json.load(file)
    annList = temp['annotations']
    for j in annList:
        cid = j['category_id']
        if(cid in categories):
            cats = temp['categories']
            
            cset = files_by_id[cid]
            cset.add(i)
            files_by_id[cid] = cset
    inc += 1
    if(inc % 1000 == 0):
        print(inc)
# The second dictionary records only files with less than 3 annotations
inc = 0
for i in files:
    with open('./cocojson/' + i, 'r+') as file:
        temp = json.load(file)
    annList = temp['annotations']
    if(len(annList) < 3):
        for j in annList:
            cid = j['category_id']
            if(cid in categories):
                cats = temp['categories']
                
                cset = files_by_id2[cid]
                cset.add(i)
                files_by_id2[cid] = cset
    inc += 1
    if(inc % 1000 == 0):
        print(inc)

In [15]:
ids2 = []
for i in files_by_id2:
    if(len(files_by_id2[i]) > 0):
        ids2.append(i)

In [17]:
target = 0
for i in files_by_id:
    if(target < len(files_by_id[i])):
        target = len(files_by_id[i])
to_do = {}
for i in files_by_id:
    to_do[i] = target - len(files_by_id[i])
sum = 0
for i in to_do:
    sum += to_do[i]
probs = np.asarray(list(to_do.values()))/sum # inverse distribution

In [ ]:
# Generates copy paste images using inverse distribution
def balance_sampling(probs, k):
    for i in range(k):
        appends = []
        for i in range(random.randint(1,30)):
            rid1 = random.choices(tuple(to_do), probs)[0]
            rfile1 = random.choice(tuple(files_by_id[rid1]))
            appends.append([rfile1, rid1])
        rfile2 = random.choice(tuple(files_by_id2[random.choice(ids2)]))
        try:
            copypaste(appends, rfile2)
        except:
            print('copypaste error')
balance_sampling(probs, 10000)